In [ ]:

import sys
import os
from pathlib import Path
import pprint

import numpy as np
import torch
from torch.utils.data import DataLoader

sys.path.append("../../_src/_001_aux_functions")
sys.path.append("../../_src/_002_readdata")
sys.path.append("../../_src/_003_ml_f")
#sys.path.append("../")

from camelsh import camelsh
from mflstm import MFLSTM
from utils import set_random_seed, Optimizer, upload_to_device


ModuleNotFoundError: No module named 'torch'

In [4]:
import subprocess
subprocess.run(["ls", "-l"])

FileNotFoundError: [WinError 2] The system cannot find the file specified

In [187]:
data_path = Path("C:/Users/mfjakows/Datasets/CAMELSH").resolve()
#data_path = Path("C:/Users/mfjakows/Downloads/tva/CAMELSH").resolve()
train_entity_path = Path("../Input/testing.txt").resolve()
#train_entity_path = Path("Input/emory_oak.txt").resolve()
#train_entity_path = Path("Input/local.txt").resolve()

In [188]:

dynamic_input = {
    "1D": ["CAPE", "CRainf_frac", "LWdown", "PotEvap", "PSurf", "Qair", "Rainf", "SWdown", "Tair", "Wind_E", "Wind_N"],
    "1h": ["CAPE", "CRainf_frac", "LWdown", "PotEvap", "PSurf", "Qair", "Rainf", "SWdown", "Tair", "Wind_E", "Wind_N"],
}
target = ["Q_camelsh_obs_norm"]
forcing = ["nldas_hourly"]
static_input = [
    "p_mean", "pet_mean", "aridity_index", "p_seasonality", "frac_snow",
    "high_prec_freq", "high_prec_dur", "low_prec_freq", "low_prec_dur",
    "ele_mt_sav", "slp_dg_uav", "ria_ha_usu", "run_mm_syr", "gwt_cm_sav",
    "cly_pc_uav", "slt_pc_uav", "snd_pc_uav", "kar_pc_use", "prm_pc_use",
    "pac_pc_use", "crp_pc_use", "for_pc_use", "urb_pc_use", "DRAIN_SQKM"
]
#training_period = ["1987-01-01 00:00:00", "2009-12-31 23:00:00"]
#validation_period = ["2010-01-01 00:00:00", "2015-12-31 23:00:00"]
#testing_period = ["2016-01-01 00:00:00", "2022-12-31 23:00:00"]
training_period = ["2018-01-01 00:00:00", "2020-12-31 23:00:00"]
testing_period = ["2022-01-01 00:00:00", "2023-12-31 23:00:00"]

lookback_window = 0
SMOKE_TEST = True
model_configuration = {
    "n_dynamic_channels_lstm": 10,
    "no_of_layers": 1,
    "seq_length": 365 * 24,
    "custom_freq_processing": {
        "1D": {"n_steps": 351, "freq_factor": 24},
        "1h": {"n_steps": (365 - 351) * 24, "freq_factor": 1},
    },
    "predict_last_n": 1,
    #"predict_last_n": (24*14 + (365-14) - 1),
    "unique_prediction_blocks": True,
    "dynamic_embeddings": True,
    "hidden_size": 32 if SMOKE_TEST else 128,
    "batch_size_training": 16 if SMOKE_TEST else 128,
    "batch_size_evaluation": 128 if SMOKE_TEST else 1024,
    "no_of_epochs": 10 if SMOKE_TEST else 30,
    "dropout_rate": 0.4,
    "learning_rate": {1: 5e-4, 10: 1e-4, 25: 1e-5},
    "set_forget_gate": 3,
    "validate_every": 1 if SMOKE_TEST else 4,
    "validate_n_random_basins": 1 if SMOKE_TEST else -1,
}

seed = 110
color_palette = {"observed": "#377eb8", "simulated": "#4daf4a"}


In [189]:
dataset = camelsh(
    dynamic_input=dynamic_input,
    target=target,
    forcing=forcing,
    sequence_length=model_configuration["seq_length"],
    #time_period=training_period,
    time_period=testing_period,
    path_data=str(data_path),
    path_entities=str(train_entity_path),
    check_NaN=True,
    predict_last_n=model_configuration["predict_last_n"],
    static_input=static_input,
    custom_freq_processing=model_configuration["custom_freq_processing"],
    dynamic_embedding=model_configuration["dynamic_embeddings"],
    unique_prediction_blocks=model_configuration["unique_prediction_blocks"],
    lookback_window=lookback_window,
)
model_configuration["dynamic_input_size"] = {key : len(value) for key, value in dynamic_input.items()}

# Use actual loaded static count (some requested columns may not exist in attribute files)
actual_static_count = dataset.df_attributes.shape[1]
model_configuration["input_size_lstm"] = model_configuration["n_dynamic_channels_lstm"] + actual_static_count
print(f"Requested static features: {len(static_input)}, Actually loaded: {actual_static_count}")
print(f"input_size_lstm: {model_configuration['input_size_lstm']}")
print(f"Loaded static columns: {list(dataset.df_attributes.columns)}")

model_configuration["predict_last_n"] = 1

device = "cuda:0" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

present = list(dataset.df_attributes.columns)


Requested static features: 24, Actually loaded: 10
input_size_lstm: 20
Loaded static columns: ['p_mean', 'pet_mean', 'aridity_index', 'p_seasonality', 'frac_snow', 'high_prec_freq', 'high_prec_dur', 'low_prec_freq', 'low_prec_dur', 'DRAIN_SQKM']


In [190]:
print(f"len dataset: {len(dataset)}")
print(f"len dataset: {len(dataset)/2}")

len dataset: 17112
len dataset: 8556.0


In [191]:

#model_configuration["dynamic_input_size"] = {key : len(value) for key, value in dynamic_input.items()}

#model_configuration["input_size_lstm"] = model_configuration["n_dynamic_channels_lstm"] + len(static_input)
#print(f"len(static_input): {len(static_input)}, len(model_configuration['n_dynamic_channels_lstm']): {(model_configuration["n_dynamic_channels_lstm"])}, model_configuration['input_size_lstm']: {model_configuration['input_size_lstm']}")
#print(model_configuration["input_size_lstm"] )

#model_configuration["predict_last_n"] = 1

#device = "cuda:0" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

In [192]:
dataset.calculate_basin_std()
dataset.calculate_global_statistics()
dataset.standardize_data()


In [193]:

train_loader = DataLoader(
    dataset=dataset,
    #batch_size=model_configuration["batch_size_training"],
    batch_size=1,
    shuffle=True,
    drop_last=True,
    collate_fn=dataset.collate_fn,
)
print("Number of batches in training:", len(train_loader))

it = (iter(train_loader))
first = next(it)
print(first)

Number of batches in training: 17112
{'x_d_1D': tensor([[[-0.3217, -0.2336, -1.9613,  ..., -1.7202,  0.4956, -1.2373],
         [-0.3158,  0.1463, -0.4228,  ..., -1.2351, -0.5341,  1.1315],
         [-0.3209, -0.2336, -0.5842,  ..., -0.9820,  0.5441, -0.1447],
         ...,
         [-0.3234, -0.2336, -1.2544,  ..., -0.9723,  0.8803, -0.7965],
         [-0.3234, -0.2336, -1.3035,  ..., -0.9212, -0.3133,  0.1132],
         [-0.3223, -0.2336, -0.7132,  ..., -0.7290, -0.8984, -0.6021]]]), 'x_d_1h': tensor([[[-0.3146, -0.2336, -0.1236,  ..., -0.8339,  0.7847, -0.6762],
         [-0.3190, -0.2336, -0.1236,  ..., -0.8233,  1.0508, -0.9081],
         [-0.3234, -0.2336, -1.0852,  ..., -0.8137,  1.3138, -1.1401],
         ...,
         [-0.3234, -0.2336, -0.2878,  ..., -1.3780, -2.3933,  0.3026],
         [-0.3234, -0.2336, -0.2878,  ..., -1.3684, -2.3841,  0.2898],
         [-0.3234, -0.2336, -0.2002,  ..., -1.3599, -2.3719,  0.2803]]]), 'x_s': tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]

In [194]:
set_random_seed(seed)
model = MFLSTM(model_configuration=model_configuration).to(device)
optimizer = Optimizer(model=model, model_configuration=model_configuration)
model.lstm.bias_hh_l0.data[model_configuration["hidden_size"]:2 * model_configuration["hidden_size"]] = model_configuration["set_forget_gate"]


sample = next(iter(train_loader))
for key, value in sample.items():
    if hasattr(value, "shape"):
        print(key, value.shape)
    else:
        print(key, type(value))
sample_device = upload_to_device(sample, device)
#print(f"hi \n {sample_device}")
print(len(sample_device["x_d_1D"][0][0]))
pprint.pprint(sample_device, depth=2)
with torch.no_grad():
    pred = model(sample_device)
print("Prediction shape:", pred["y_sim"].shape)

x_d_1D torch.Size([1, 351, 11])
x_d_1h torch.Size([1, 336, 11])
x_s torch.Size([1, 10])
y_obs torch.Size([1, 1, 1])
basin_std torch.Size([1, 1, 1])
basin (1,)
date (1, 1)
11
{'basin': array(['01123000'], dtype='<U8'),
 'basin_std': tensor([[[3.2525]]], device='cuda:0'),
 'date': array([['2023-10-17T02:00:00.000000']], dtype='datetime64[us]'),
 'x_d_1D': tensor([[[-0.2897,  0.9200,  0.2033,  ...,  0.1222, -0.5902,  0.5417],
         [-0.3092,  0.3979,  0.3894,  ...,  0.1420,  0.2319,  0.5062],
         [-0.3217, -0.2336, -1.0167,  ..., -0.4020,  0.8605,  0.6912],
         ...,
         [-0.3139, -0.1587,  0.9519,  ...,  0.3482, -1.3463, -1.1017],
         [-0.3234, -0.2336, -0.0351,  ...,  0.5120, -0.6235, -1.1829],
         [-0.3234, -0.2336,  0.2415,  ...,  0.7154, -0.5488, -0.7057]]],
       device='cuda:0'),
 'x_d_1h': tensor([[[-0.3234, -0.2336, -0.3805,  ...,  0.3043,  0.2647,  0.5663],
         [-0.3234, -0.2336, -0.3804,  ...,  0.2254,  0.3809,  0.4074],
         [-0.3234, -0.23

In [195]:
print("model.lstm.input_size:", model.lstm.input_size)
print("configured input_size_lstm:", model_configuration["input_size_lstm"])
print("len(static_input requested):", len(static_input))

print("x_d_1D:", sample_device["x_d_1D"].shape)
print("x_d_1h:", sample_device["x_d_1h"].shape)
print("x_s:", sample_device["x_s"].shape)

# Rebuild exactly what MFLSTM does
x1 = model.embedding_net["1D"](sample_device["x_d_1D"])
x2 = model.embedding_net["1h"](sample_device["x_d_1h"])
x = torch.cat([x1, x2], dim=1)
print("after dynamic embedding:", x.shape)            # last dim should be 10

x = torch.cat((x, sample_device["x_s"].unsqueeze(1).repeat(1, x.shape[1], 1)), dim=2)
print("after static concat:", x.shape)                # this last dim is what LSTM receives

model.lstm.input_size: 20
configured input_size_lstm: 20
len(static_input requested): 24
x_d_1D: torch.Size([1, 351, 11])
x_d_1h: torch.Size([1, 336, 11])
x_s: torch.Size([1, 10])
after dynamic embedding: torch.Size([1, 687, 10])
after static concat: torch.Size([1, 687, 20])
